# SDLC BVE Dashboards for X - Getting Started in Codespaces

Use this notebook as the fastest guided path from a fresh Codespace to a working local setup.

## What this repo does

- Collects GitHub Copilot, PR, and agentic coding data
- Materializes pipeline artifacts into `dashboard/dataflow/data/`
- Publishes browser-only dashboards through GitHub Pages

## Current paths to prefer

- Use `.github/workflows/pipeline-deploy.yml` as the current pipeline workflow
- Treat `.github/workflows/deploy-dashboards.yml` as a legacy workflow
- Prefer current V4 dashboards when they exist, especially:
  - `dashboard/v4/ai-assisted-efficiency/`
  - `dashboard/v4/agentic-efficiency/`
- Treat older manual-upload or older-version dashboard paths as compatibility or legacy paths unless you are intentionally working there

## Source-of-truth docs

- `README.md`
- `docs/getting-started.md`
- `docs/pat-setup.md`
- `docs/data-collection.md`
- `docs/dashboard-status.md`
- `dependencies/README.md`


In [ ]:
!pwd
!node --version
!npm --version
!python3 --version
!gh --version | head -n 1
!copilot --version || true
!jupyter lab --version || true

## 1. Install dependencies

Most Codespaces/devcontainer installs will already run the post-create setup, but this is safe to re-run if needed.

In [ ]:
!npm install --include=dev

## 2. Set up a GitHub PAT

Create a Personal Access Token (classic) with these scopes:

- `copilot` - always required for Copilot metrics
- `read:org` - org-level data
- `read:enterprise` - enterprise-level data
- `repo` - PR metrics and Actions logs

If your organization uses SAML SSO, authorize the token for the org after creating it.

See `docs/pat-setup.md` for the full click-by-click flow.

## 3. Configure the repo

Edit these files before running the pipeline:

### `query-settings.json`

```json
{
  "default": {
    "ORG": "your-org",
    "ENTERPRISE": "your-enterprise-slug",
    "DAYS": "28"
  }
}
```

### `dashboard-config.json`

Set at least:

- `cfg_total_developers`
- `cfg_pct_time_coding`
- `cfg_labor_cost_per_hour`
- `est_hrs_per_kloc`
- `est_duration_factor`

Getting `cfg_total_developers` right matters because many adoption and efficiency percentages depend on it.

In [ ]:
!sed -n '1,80p' query-settings.json
!echo '---'
!sed -n '1,120p' dashboard-config.json

## 4. Authenticate

You can either use the GitHub CLI session or export a token directly.

### Option A - GitHub CLI

```bash
gh auth login
```

### Option B - Environment variable

```bash
export GITHUB_TOKEN="ghp_your_token_here"
```

For GitHub Actions / Pages deployment, set the repository secret `DASHBOARD_GH_TOKEN` and repository variables like `ORG`, `ENTERPRISE`, and `DAYS`.

## 5. Verify configuration before collecting data

This is the safest first command. It resolves config and shows what the pipeline will do without making API calls.

In [ ]:
!./run-query.sh --dry-run

## 6. Run the pipeline locally

Use `run-query.sh` from the repo root.

Common commands:

- `./run-query.sh`
- `./run-query.sh --materialize-only`
- `./run-query.sh --session-logs-only`
- `DAYS=7 ./run-query.sh`


In [ ]:
!echo 'Uncomment and run one of these when ready:'
!echo './run-query.sh'
!echo './run-query.sh --materialize-only'
!echo './run-query.sh --session-logs-only'

## 7. Serve dashboards locally

Run the local server and open dashboards from the `dashboard/` tree.

In [ ]:
!echo 'Start the local server in a terminal:'
!echo 'npm run serve'
!echo 'Then open http://127.0.0.1:8080/'

## 8. Deploy to GitHub Pages

Current workflow:

- `.github/workflows/pipeline-deploy.yml`

Recommended trigger:

```bash
gh workflow run pipeline-deploy.yml
```

If you only want to redeploy from existing data:

```bash
gh workflow run pipeline-deploy.yml -f skip_data_collection=true
```


## 9. Troubleshooting

### PAT / auth problems

- `401 Unauthorized` -> token missing or expired
- `403 Forbidden` on org data -> token not SSO-authorized for the org
- `404` on enterprise endpoints -> missing `read:enterprise`
- empty Copilot metrics -> missing `copilot` scope

### Pipeline / data problems

- Run `./run-query.sh --dry-run`
- Re-check `ORG`, `ENTERPRISE`, and `DAYS`
- Re-run `./run-query.sh --materialize-only` if raw data exists but artifacts are stale

### Codespaces / Jupyter problems

- Rebuild the devcontainer after devcontainer changes
- Confirm `jupyter lab --version` works
- Confirm the Python kernel named `Python 3 (BVE Dashboards)` is available

### Understanding current vs legacy paths

- Prefer V4 dashboard paths when available
- Prefer `.github/workflows/pipeline-deploy.yml`
- Treat older workflows and older manual-upload dashboards as legacy unless your task is specifically about them


In [ ]:
!echo 'Helpful docs:'
!printf '%s\n' README.md docs/getting-started.md docs/pat-setup.md docs/data-collection.md docs/dashboard-status.md dependencies/README.md